# 7)  Business Reporting Layer — Geo Incrementality Experiment
### Spillover-Aware Measurement Pipeline · Decision-Grade Output

---

## What this notebook does

This notebook is the **final layer** of a production-style geo incrementality
measurement pipeline. It takes validated causal estimates from the estimation
notebook and answers the only question that matters to a business stakeholder:

> **Did paid social drive incremental revenue — and was the spend worth it?**

It does this in three steps:

1. **Selects a primary estimator** using a rule-based validity gate (spillover
   contamination, pre-trend violations, SCM pre-fit quality, CI availability).
   The selection logic is explicit and auditable — not a judgment call.

2. **Translates lift into business KPIs**: incremental revenue, iROAS with 95%
   confidence bounds, ROI, cost per incremental unit, and break-even probability.
   All financial metrics inherit uncertainty directly from the statistical CI —
   there are no point-estimate-only outputs.

3. **Issues a gated recommendation** (Scale / Hold / Stop) with a full risk flag
   register: pre-trend status, spillover contamination level, SCM fit quality,
   and cross-model consensus. The recommendation cannot be SCALE if any
   hard trust gate is failing.

---

## Design principles

| Principle | Implementation |
|---|---|
| No hardcoded spend | `spend_total` always read from `df.paid_social_spend` |
| No silent failures | Every flag is computed programmatically, not asserted manually |
| Uncertainty propagates | CI endpoints transformed through iROAS, not just the point estimate |
| Estimator selection is auditable | Rule table printed with explicit exclusion reasons |
| Corroboration is mandatory | Primary estimate reported alongside Synth + Bayes consensus range |

---

## Inputs consumed

| Input | Source | Description |
|---|---|---|
| `df` | `data/simulated_geo_panel.csv` | Daily panel: market × date, includes `paid_social_spend`, `sales`, `sales_baseline`, `spillover_risk_group` |
| `summary` | `results/estimator_summary.csv` | Estimator outputs with trust metrics + validity flags from `01_causal_estimation.ipynb` |

---

## Outputs produced

| Output | Type | Used for |
|---|---|---|
| `primary_estimator` | string | Identifies which model drives the decision |
| `metrics_point/low/high` | dict | iROAS, ROI, CPIU at point + CI bounds |
| `breakeven_p` | float | P(iROAS > 1) — confidence the campaign is net positive |
| Decision table | HTML | Stakeholder-facing recommendation with full risk register |

---

## Assumptions

- `unit_value = 1.0` — **one unit of `sales` equals $1 of revenue.** This is a
  simulation convention. In a real deployment, replace with average order value
  or revenue-per-unit from your data warehouse.
- `gross_margin = 1.0` — **no COGS applied.** Set to e.g. `0.40` for a 40% margin
  product to get profit-based ROI.
- Bootstrap CIs (300 reps, block bootstrap over markets) were computed in the
  estimation notebook. P(iROAS > 1) is approximated via
  Normal(lift_mean, lift_se) where lift_se is derived from the 95% CI width —
  valid by CLT given ≥ 300 bootstrap replicates.

---

## How to run
```bash
# From repo root
PYTHONPATH=src jupyter notebook notebooks/02_business_reporting.ipynb
```

Run cells top-to-bottom. All dependencies are loaded from saved CSVs —
this notebook does not need the estimation notebook to be open or in memory.
If flags or SCM metrics are missing, cell 1 will recompute them automatically
from `df` using the same logic as the estimation notebook.

In [18]:
from pathlib import Path
import os, sys
import pandas as pd
import numpy as np
from scipy import stats

REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..")) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
SRC_PATH = os.path.join(REPO_ROOT, "src")
if SRC_PATH not in sys.path:
    sys.path.insert(0, SRC_PATH)
    
np.random.seed(7)

In [43]:
# import reporting functions from reporting_metrics.py
from geo_experiment.reporting_metrics import compute_experiment_totals, reporting_metrics_from_lift

In [47]:
# Load the estimator summary
df = pd.read_csv("../data/simulated_geo_panel.csv", parse_dates=["date"])
summary = pd.read_csv("../results/estimator_summary.csv")
print("Loaded estimator summary:")
display(summary)

Loaded estimator summary:


,estimator,lift_hat_total,trust_metric_1,trust_metric_2,valid,ci_low,ci_high,ci_width,reason_invalid
0,DiD (pre/post sums),-49607.228442,pretrend_flag=False,spillover_flag=True,False,-690739.072582,590574.651817,1.281314e+06,spillover_flag=True (control contamination risk)
1,TBR/GBR (controls OLS),76636.297408,r2_controls=0.994,spillover_flag=True,False,34156.706858,126165.397657,9.200869e+04,spillover_flag=True (control contamination risk)
2,CUPED (variance reduction),82518.901978,theta=0.363,var_red=99.4%,True,36712.459719,135163.197220,9.845074e+04,NaN
3,CUPAC (aux prediction),73881.469156,r2_controls=0.994,spillover_flag=True,False,23370.238831,122826.752097,9.945651e+04,spillover_flag=True (control contamination risk)
4,Synth (all controls),87713.795008,prefit_rmspe=1998,loo_rel_std=0.015,True,NaN,NaN,NaN,NaN
5,Bayes hier (partial pooling),93575.401220,tau=1095.4,shrink_mean=0.87,True,NaN,NaN,NaN,NaN


## 7.1 Pick a “Primary” estimator (rule-based)

Don’t just choose Bayes because it’s best in sim — define decision logic:

Example rule:

- If spillover flag → DiD invalid
- If SCM prefit RMSPE too high or `w_max≈1` → SCM invalid
- If CUPAC R² < threshold → downweight
- Choose among remaining: CUPED/CUPAC/TBR/Bayes (rank by validity + CI tightness)

Output:

- `primary_estimator = ...`
- short explanation text

In [5]:
# ============================================================
# 7.1 Pick a "Primary" Estimator (rule-based)
# ============================================================

def pick_primary_estimator(summary: pd.DataFrame) -> tuple[pd.Series, str]:
    """
    Rule-based logic to select the primary estimator:
    1. Exclude if spillover_flag=True  → control contamination risk
    2. Exclude if SCM prefit_rmspe too high or w_max≈1 → SCM invalid
    3. Downweight if CUPAC R² < threshold
    4. Among remaining valid: rank by validity + CI tightness (ci_width)
    """
    df = summary.copy()

    # Rule 1: Drop spillover-flagged estimators
    def has_spillover(row):
        return (
            (isinstance(row.get("trust_metric_1"), str) and "spillover_flag=True" in str(row["trust_metric_1"])) or
            (isinstance(row.get("trust_metric_2"), str) and "spillover_flag=True" in str(row["trust_metric_2"]))
        )

    df["_spillover"] = df.apply(has_spillover, axis=1)

    # Rule 2: Drop SCM if prefit_rmspe too high (heuristic: > 100k) or w_max≈1
    def scm_invalid(row):
        if "synth" not in str(row["estimator"]).lower():
            return False
        tm1 = str(row.get("trust_metric_1", ""))
        if "prefit_rmspe" in tm1:
            try:
                rmspe_val = float(tm1.split("=")[1])
                if rmspe_val > 100_000:
                    return True
            except Exception:
                pass
        return False

    df["_scm_invalid"] = df.apply(scm_invalid, axis=1)

    # Rule 3: CUPAC R² threshold (downweight but don't exclude)
    def cupac_low_r2(row):
        if "cupac" not in str(row["estimator"]).lower():
            return False
        tm1 = str(row.get("trust_metric_1", ""))
        if "r2_controls" in tm1:
            try:
                r2_val = float(tm1.split("=")[1])
                return r2_val < 0.8
            except Exception:
                pass
        return False

    df["_cupac_low_r2"] = df.apply(cupac_low_r2, axis=1)

    # Rule 4: Filter to valid candidates
    candidates = df[
        (df["valid"] == True) &
        (~df["_spillover"]) &
        (~df["_scm_invalid"])
    ].copy()

    if candidates.empty:
        # Fallback: pick best valid estimator ignoring flags
        candidates = df[df["valid"] == True].copy()

    # Among candidates, rank by CI width (ascending = tighter = better)
    # Estimators with NaN ci_width (e.g. Synth, Bayes) are ranked lower
    candidates["_ci_width_rank"] = candidates["ci_width"].rank(na_option="bottom")
    candidates = candidates.sort_values("_ci_width_rank")

    primary_row = candidates.iloc[0]
    primary_name = primary_row["estimator"]

    # Build explanation
    excluded = []
    for _, row in df.iterrows():
        if row["_spillover"]:
            excluded.append(f"• {row['estimator']}: excluded — spillover contamination risk")
        elif row["_scm_invalid"]:
            excluded.append(f"• {row['estimator']}: excluded — SCM prefit RMSPE too high")
        elif row["valid"] != True:
            excluded.append(f"• {row['estimator']}: excluded — marked invalid ({row.get('reason_invalid', '')})")

    explanation = (
        f"Primary estimator selected: **{primary_name}**\n"
        f"Reason: Valid, no spillover flag, tightest CI (ci_width = {primary_row['ci_width']:.0f}).\n\n"
        "Excluded estimators:\n" + "\n".join(excluded)
    )

    print(explanation)
    return primary_row, primary_name


primary_row, primary_estimator = pick_primary_estimator(summary)
print(f"\nprimary_estimator = '{primary_estimator}'")

Primary estimator selected: **CUPED (variance reduction)**
Reason: Valid, no spillover flag, tightest CI (ci_width = 98451).

Excluded estimators:
• DiD (pre/post sums): excluded — spillover contamination risk
• TBR/GBR (controls OLS): excluded — spillover contamination risk
• CUPAC (aux prediction): excluded — spillover contamination risk

primary_estimator = 'CUPED (variance reduction)'


## Why CUPED is selected as primary over Bayes hierarchical

Bayes hierarchical produces the most accurate point estimate (93,575 — only 2.2% from
ground truth), and in a real experiment it would be the strongest overall model.
**However, primary estimator selection is not purely about point accuracy.**
It requires satisfying three criteria simultaneously:

| Criterion | CUPED (Variance Reduction) | Bayes Hier (Partial Pooling) |
| :--- | :--- | :--- |
| **Statistical Validity** | ✅ Pass (All trust gates) | ✅ Pass (All trust gates) |
| **Uncertainty Quantification** | ✅ 95% Bootstrap CI | ❌ NaN (Point estimate only) |
| **Business Utility** | ✅ High (Computable iROAS bounds) | ❌ Low (Risk-unaware reporting) |
| **Error vs. Ground Truth** | ⚠️ $-9.9\%$ | ✅ $+2.2\%$ |



### The CI problem with Bayes here

Without a credible interval, you cannot:
- Compute `iROAS_low` / `iROAS_high` for budget decisions
- Calculate `P(iROAS > 1)` (break-even probability)
- Report statistical uncertainty to stakeholders
- Know whether the 93,575 estimate is precise or just lucky

A point estimate without uncertainty bounds is not a complete inference result.
In production, shipping a lift number without a CI is equivalent to saying
*"we think the campaign made money, but we can't quantify how confident we are."*
That is not sufficient for a spend decision.

#### The "So What?" of Bayesian Accuracy: Engineering Trade-offs
A common question here is: *"If you knew Bayes was more accurate, why not use it as the primary?"*
- **Production Latency:**  Generating credible intervals for Hierarchical models via MCMC or bootstrapping $300\times$ introduces **significant compute overhead and latency** for daily automated reporting.
- **Decision-Grade vs. Precision-Grade:**  In causal inference, a precise, valid estimate with narrow bounds (CUPED's $99.4\%$ variance reduction) is often superior to a "lucky" point estimate without a range.
- **The Roadmap (Phase 2):**  To reach the senior DS bar, the next iteration will replace the frequentist bootstrap with **Posterior Credible Intervals for the Bayesian model**, combining its superior accuracy with robust uncertainty quantification.
### The right way to think about this

- **CUPED is the primary estimator** — it is the inference workhorse.
  Its CI [36.7K, 135.2K] is the formal uncertainty statement for this experiment.

- **Bayes and Synth are corroborating signals** — they validate that CUPED's
  point estimate is in the right ballpark. The three valid methods cluster
  within 11K of each other (82.5K / 87.7K / 93.6K), which is strong triangulation.

### Decision
> **Primary: CUPED (82,519 | CI [36,712 — 135,163])**
> Corroborated by Synth (87,714) and Bayes (93,575).
> Consensus range: **83K – 94K incremental sales.**

## 7.2 Compute spend totals + iROAS with CI (1–2 cells)

For your primary estimator:

- `lift_hat_total`, `CI_low`, `CI_high`
- spend = treated post spend total
- iROAS = $(Lift * Unit Value)/Spend$
- iROAS CI = transform lift CI endpoints

Show a compact table:

- lift [CI]
- spend
- iROAS [CI]
- break-even probability: `P(iROAS > 1)` (optional if you approximate via bootstrap draws)

In [49]:
# ── 7.2 Spend totals + iROAS with CI ─────────────────────────────────────────
# Primary estimator: CUPED (only valid CI-bearing method)
# Corroborated by: Synth (87,714) and Bayes (93,575)
# ─────────────────────────────────────────────────────────────────────────────

# ── Pull primary estimator values from final summary table ───────────────────
primary_row      = summary.loc[summary["estimator"] == "CUPED (variance reduction)"].iloc[0]
primary_lift     = float(primary_row["lift_hat_total"])
primary_ci_low   = float(primary_row["ci_low"])
primary_ci_high  = float(primary_row["ci_high"])

# ── Experiment totals (spend + sales from df) ─────────────────────────────────
totals = compute_experiment_totals(df)
spend_total = totals["spend_treated_post"]

print("── Experiment totals ───────────────────────────────────────")
print(f"   Treated post spend : ${spend_total:>12,.2f}")
print(f"   Treated post sales : ${totals['sales_treated_post']:>12,.2f}")
print(f"   Control post sales : ${totals['sales_ctrl_post']:>12,.2f}")
print(f"   Treated markets    : {totals['n_treated_markets']}")
print(f"   Control markets    : {totals['n_ctrl_markets']}")

── Experiment totals ───────────────────────────────────────
   Treated post spend : $  945,699.90
   Treated post sales : $1,947,841.76
   Control post sales : $4,922,399.77
   Treated markets    : 16
   Control markets    : 44


In [55]:
# ── Point estimate metrics ────────────────────────────────────────────────────
metrics_point = reporting_metrics_from_lift(
    lift_hat_total = primary_lift,
    spend_total    = spend_total,
    unit_value     = 1.0,    # 1 sales unit = $1 revenue; adjust if units differ
    gross_margin   = 1.0,    # set e.g. 0.4 if you want profit-based ROI
    fixed_cost     = 0.0,    # add fixed campaign costs here if applicable
)

# ── CI bound metrics (transform lift CI endpoints through same function) ──────
metrics_low = reporting_metrics_from_lift(
    lift_hat_total = primary_ci_low,
    spend_total    = spend_total,
    unit_value     = 1.0,
    gross_margin   = 1.0,
    fixed_cost     = 0.0,
)

metrics_high = reporting_metrics_from_lift(
    lift_hat_total = primary_ci_high,
    spend_total    = spend_total,
    unit_value     = 1.0,
    gross_margin   = 1.0,
    fixed_cost     = 0.0,
)

# ── Break-even probability: P(iROAS > 1) ≡ P(lift > spend) ──────────────────
# Approximate via Normal(lift_mean, lift_se) — valid given 300 bootstrap reps
lift_se     = (primary_ci_high - primary_ci_low) / (2 * 1.96)
breakeven_p = float(1 - stats.norm.cdf(spend_total, loc=primary_lift, scale=lift_se))

# ── Corroborating iROAS from Synth + Bayes (point only, no CI) ───────────────
synth_lift = float(summary.loc[summary["estimator"] == "Synth (all controls)",          "lift_hat_total"].iloc[0])
bayes_lift = float(summary.loc[summary["estimator"] == "Bayes hier (partial pooling)",  "lift_hat_total"].iloc[0])
iroas_synth = synth_lift / spend_total if spend_total > 0 else float("nan")
iroas_bayes = bayes_lift / spend_total if spend_total > 0 else float("nan")

# ── Decision table ────────────────────────────────────────────────────────────
decision_table = pd.DataFrame([
    {
        "metric":  "Estimator",
        "point":   "CUPED (primary)",
        "ci":      "Only valid CI-bearing method",
    },
    {
        "metric":  "Incremental lift (units)",
        "point":   f"{primary_lift:,.0f}",
        "ci":      f"[{primary_ci_low:,.0f}  —  {primary_ci_high:,.0f}]",
    },
    {
        "metric":  "Incremental revenue ($)",
        "point":   f"${metrics_point['incremental_revenue']:,.0f}",
        "ci":      f"[${metrics_low['incremental_revenue']:,.0f}  —  ${metrics_high['incremental_revenue']:,.0f}]",
    },
    {
        "metric":  "Total ad spend ($)",
        "point":   f"${spend_total:,.0f}",
        "ci":      "—",
    },
    {
        "metric":  "iROAS  (revenue / spend)",
        "point":   f"{metrics_point['iROAS']:.3f}x",
        "ci":      f"[{metrics_low['iROAS']:.3f}x  —  {metrics_high['iROAS']:.3f}x]",
    },
    {
        "metric":  "ROI  ((profit − spend) / spend)",
        "point":   f"{metrics_point['ROI']:.3f}",
        "ci":      f"[{metrics_low['ROI']:.3f}  —  {metrics_high['ROI']:.3f}]",
    },
    {
        "metric":  "Cost per incremental unit ($)",
        "point":   f"${metrics_point['cost_per_incremental_unit']:.2f}",
        "ci":      f"[${metrics_high['cost_per_incremental_unit']:.2f}  —  ${metrics_low['cost_per_incremental_unit']:.2f}]",
    },
    {
        "metric":  "P(iROAS > 1)  [break-even prob]",
        "point":   f"{breakeven_p:.1%}",
        "ci":      "Normal approx via bootstrap SE",
    },
    {
        "metric":  "Corroborating iROAS — Synth",
        "point":   f"{iroas_synth:.3f}x",
        "ci":      "No CI (use as signal only)",
    },
    {
        "metric":  "Corroborating iROAS — Bayes",
        "point":   f"{iroas_bayes:.3f}x",
        "ci":      "No CI (use as signal only)",
    },
    {
        "metric":  "Decision",
        "point":   "SCALE ✅" if metrics_point["iROAS"] > 1.0 else "PAUSE ⚠️",
        "ci":      "iROAS > 1.0 = campaign net positive",
    },
]).set_index("metric")

display(decision_table)

print(f"\n── Interpretation ──────────────────────────────────────────")
print(f"   Every $1 of ad spend generated ${metrics_point['iROAS']:.2f} in incremental revenue (CUPED).")
print(f"   The 95% CI [{metrics_low['iROAS']:.2f}x — {metrics_high['iROAS']:.2f}x] stays {'above' if metrics_low['iROAS'] > 1 else 'below'} break-even at the lower bound.")
print(f"   Break-even probability: {breakeven_p:.1%}.")
print(f"   Synth ({iroas_synth:.2f}x) and Bayes ({iroas_bayes:.2f}x) corroborate the direction.")

,point,ci
metric,,
Estimator,CUPED (primary),Only valid CI-bearing method
Incremental lift (units),"82,519","[36,712 — 135,163]"
Incremental revenue ($),"$82,519","[$36,712 — $135,163]"
Total ad spend ($),"$945,700",—
iROAS (revenue / spend),0.087x,[0.039x — 0.143x]
ROI ((profit − spend) / spend),-0.913,[-0.961 — -0.857]
Cost per incremental unit ($),$11.46,[$7.00 — $25.76]
P(iROAS > 1) [break-even prob],0.0%,Normal approx via bootstrap SE
Corroborating iROAS — Synth,0.093x,No CI (use as signal only)



── Interpretation ──────────────────────────────────────────
   Every $1 of ad spend generated $0.09 in incremental revenue (CUPED).
   The 95% CI [0.04x — 0.14x] stays below break-even at the lower bound.
   Break-even probability: 0.0%.
   Synth (0.09x) and Bayes (0.10x) corroborate the direction.


## 7.2.1 Sensitivity Analysis: Unit Economics & P&L Impact
**Strategic Sensitivity:** At what AOV/Margin does this spend scale?
The current campaign "failed" because the Cost Per Incremental Unit (11.46USD) exceeded the Unit Value (1.00USD). Below is a sensitivity analysis showing the Average Order Value (AOV) required to reach profitability at current performance levels.

In [85]:
def generate_sensitivity_table(lift, spend, unit_values, margins):
    """
    Generates a grid showing iROAS or Profitability across different 
    business contexts (AOV and Margins).
    """
    results = []
    for aov in unit_values:
        row = {}
        for m in margins:
            # iROAS = (Lift * AOV) / Spend
            # We look for the break-even point where iROAS * Margin > 1
            iroas = (lift * aov) / spend
            effective_roi = (iroas * m) - 1
            
            status = "🟢 SCALE" if effective_roi > 0 else "🔴 STOP"
            row[f"Margin {int(m*100)}%"] = f"{iroas:.2f}x ({status})"
        row["Average Order Value (AOV)"] = f"${aov}"
        results.append(row)
    
    return pd.DataFrame(results).set_index("Average Order Value (AOV)")

# Define scenarios relevant to your product category
potential_aovs = [1.0, 5.0, 10.0, 15.0, 20.0] # Current is 1.0
potential_margins = [0.4, 0.6, 0.8, 1.0]      # Current is 1.0

sensitivity_df = generate_sensitivity_table(
    lift=primary_lift, 
    spend=spend_total, 
    unit_values=potential_aovs, 
    margins=potential_margins
)

print("### Strategic Sensitivity: At what AOV/Margin does this spend make sense?")
display(sensitivity_df)

### Strategic Sensitivity: At what AOV/Margin does this spend make sense?


,Margin 40%,Margin 60%,Margin 80%,Margin 100%
Average Order Value (AOV),,,,
$1.0,0.09x (🔴 STOP),0.09x (🔴 STOP),0.09x (🔴 STOP),0.09x (🔴 STOP)
$5.0,0.44x (🔴 STOP),0.44x (🔴 STOP),0.44x (🔴 STOP),0.44x (🔴 STOP)
$10.0,0.87x (🔴 STOP),0.87x (🔴 STOP),0.87x (🔴 STOP),0.87x (🔴 STOP)
$15.0,1.31x (🔴 STOP),1.31x (🔴 STOP),1.31x (🟢 SCALE),1.31x (🟢 SCALE)
$20.0,1.75x (🔴 STOP),1.75x (🟢 SCALE),1.75x (🟢 SCALE),1.75x (🟢 SCALE)


### Strategic Interpretation
- **Break-Even Thresholds:** The campaign only reaches a "SCALE" status when the AOV is 15.0USD or higher with at least an 80% gross margin. At an AOV of 20.0USD, the campaign becomes viable at a lower 60% margin.

- **Unit Economics Misalignment:** For the current simulation (AOV = 1.0USD), the iROAS of 0.09x is significantly below the 1.0x break-even line regardless of the margin. This indicates that the cost per incremental unit (11.46USD) is currently over 11 times higher than the revenue generated per unit.

- **Actionable Strategy:** This table suggests that the paid social creative or targeting used in this experiment is better suited for high-ticket items rather than low-cost units. To make this campaign successful for the current $1.0 product, we would need to either reduce the cost-per-acquisition by ~90% or significantly increase the customer's lifetime value/AOV.

## 7.3 “Decision Table”

A single table with:

- **Recommendation**: Scale / Hold / Stop
- Why (2–3 bullets)
- Risk flags:
    - pretrend warning?
    - spillover contamination?
    - SCM invalid?
    - model mismatch?

This is what a hiring manager wants.

In [75]:
# ── 7.3 Pre-requisites: recompute all flags + values needed for decision table
# Run this cell first if executing the reporting notebook independently
# ─────────────────────────────────────────────────────────────────────────────

import numpy as np
import pandas as pd
from scipy import stats
from IPython.display import display, HTML

# ── Load summary table if not already in memory ───────────────────────────────
# try:
    # final
    # print("✅ 'final' summary table found in memory")
# except NameError:
    # final = pd.read_csv("../results/estimator_summary.csv")
    # print("📂 Loaded 'final' from results/estimator_summary.csv")

# ── Load df if not already in memory ─────────────────────────────────────────
try:
    df
    print("✅ 'df' panel found in memory")
except NameError:
    df = pd.read_csv("../data/simulated_geo_panel.csv", parse_dates=["date"])
    print("📂 Loaded 'df' from data/simulated_geo_panel.csv")

# ── Recompute flag_pretrend ───────────────────────────────────────────────────
try:
    flag_pretrend
    print("✅ flag_pretrend found in memory")
except NameError:
    from geo_experiment.diagnostics.pretrend import pretrend_slope_test
    pretrend_res  = pretrend_slope_test(df, outcome_col="sales", log=True, alpha=0.05)
    flag_pretrend = bool(pretrend_res.flag_pretrend)
    print(f"🔁 Recomputed flag_pretrend = {flag_pretrend}")

# ── Recompute flag_spillover ──────────────────────────────────────────────────
try:
    flag_spillover
    print("✅ flag_spillover found in memory")
except NameError:
    ctrl_post = df.loc[(df["is_test"] == 0) & (df["is_post"] == 1)].copy()
    ctrl_post["excess_over_baseline"] = ctrl_post["sales"] - ctrl_post["sales_baseline"]
    spillover_summary = (
        ctrl_post
        .groupby("spillover_risk_group", as_index=False)
        .agg(mean_excess=("excess_over_baseline", "mean"))
    )
    flag_spillover = bool((spillover_summary["mean_excess"] > 0).any())
    print(f"🔁 Recomputed flag_spillover = {flag_spillover}")

# ── Recompute SCM trust metrics ───────────────────────────────────────────────
try:
    synth_all_fit
    print("✅ synth_all_fit found in memory")
except NameError:
    from geo_experiment.estimators.synth import aggregate_synth_control
    all_controls  = sorted(df.loc[df["is_test"] == 0, "market_id"].unique())
    synth_all_fit = aggregate_synth_control(df, donor_markets=all_controls, constrained=True)
    print(f"🔁 Recomputed synth_all_fit  (lift={synth_all_fit['lift_hat_total']:,.0f})")

# ── Recompute primary estimator values ────────────────────────────────────────
try:
    primary_lift
    print("✅ primary estimator values found in memory")
except NameError:
    primary_row    = summary.loc[final["estimator"] == "CUPED (variance reduction)"].iloc[0]
    primary_lift   = float(primary_row["lift_hat_total"])
    primary_ci_low = float(primary_row["ci_low"])
    primary_ci_high= float(primary_row["ci_high"])
    print(f"🔁 Recomputed primary_lift={primary_lift:,.0f}  CI=[{primary_ci_low:,.0f}, {primary_ci_high:,.0f}]")

# ── Recompute spend + iROAS metrics ──────────────────────────────────────────
try:
    metrics_point
    print("✅ metrics_point found in memory")
except NameError:
    totals      = compute_experiment_totals(df)
    spend_total = totals["spend_treated_post"]
    metrics_point = reporting_metrics_from_lift(primary_lift,     spend_total)
    metrics_low   = reporting_metrics_from_lift(primary_ci_low,   spend_total)
    metrics_high  = reporting_metrics_from_lift(primary_ci_high,  spend_total)
    lift_se       = (primary_ci_high - primary_ci_low) / (2 * 1.96)
    breakeven_p   = float(1 - stats.norm.cdf(spend_total, loc=primary_lift, scale=lift_se))
    print(f"🔁 Recomputed iROAS={metrics_point['iROAS']:.3f}x  P(iROAS>1)={breakeven_p:.1%}")

# ── Ground truth (simulation only) ───────────────────────────────────────────
try:
    gt_total
    print("✅ gt_total found in memory")
except NameError:
    gt_total = float(
        df.loc[(df["is_post"] == 1) & (df["is_test"] == 1), "ground_truth_lift"].sum()
    )
    print(f"🔁 Recomputed gt_total={gt_total:,.0f}")

# ── Corroborating lifts ───────────────────────────────────────────────────────
synth_lift = float(final.loc[final["estimator"] == "Synth (all controls)",         "lift_hat_total"].iloc[0])
bayes_lift = float(final.loc[final["estimator"] == "Bayes hier (partial pooling)", "lift_hat_total"].iloc[0])

print("\n✅ All dependencies ready — safe to run the decision table cell.")

✅ 'df' panel found in memory
✅ flag_pretrend found in memory
✅ flag_spillover found in memory
✅ synth_all_fit found in memory
✅ primary estimator values found in memory
✅ metrics_point found in memory
✅ gt_total found in memory

✅ All dependencies ready — safe to run the decision table cell.


In [65]:
# ── 7.3 Decision Table ────────────────────────────────────────────────────────
# A single-cell executive summary: recommendation + risk flags
# This is what goes in the stakeholder slide.
# ─────────────────────────────────────────────────────────────────────────────

from IPython.display import display, HTML

# ── 1) Recommendation logic ───────────────────────────────────────────────────
iroas_point = metrics_point["iROAS"]
iroas_low   = metrics_low["iROAS"]

if iroas_point > 1.0 and iroas_low > 0.5 and breakeven_p >= 0.80:
    recommendation  = "SCALE ✅"
    rec_color       = "#1B5E20"
    rec_bg          = "#E8F5E9"
    why = [
        f"iROAS = {iroas_point:.2f}x — campaign generated ${iroas_point:.2f} for every $1 spent.",
        f"95% CI [{iroas_low:.2f}x — {metrics_high['iROAS']:.2f}x] — lower bound remains above break-even.",
        f"Break-even probability = {breakeven_p:.1%} — high confidence the campaign is net positive.",
    ]
elif iroas_point > 1.0 and (iroas_low <= 0.5 or breakeven_p < 0.80):
    recommendation  = "HOLD ⚠️"
    rec_color       = "#E65100"
    rec_bg          = "#FFF3E0"
    why = [
        f"iROAS point estimate = {iroas_point:.2f}x is positive, but CI is wide.",
        f"Lower CI bound = {iroas_low:.2f}x — meaningful probability of sub-breakeven outcome.",
        f"Break-even probability = {breakeven_p:.1%} — insufficient confidence to scale spend.",
    ]
else:
    recommendation  = "STOP ❌"
    rec_color       = "#B71C1C"
    rec_bg          = "#FFEBEE"
    why = [
        f"iROAS = {iroas_point:.2f}x — campaign did not recover its spend.",
        f"95% CI [{iroas_low:.2f}x — {metrics_high['iROAS']:.2f}x] — wide and includes strongly negative region.",
        f"Break-even probability = {breakeven_p:.1%} — insufficient evidence of positive return.",
    ]

# ── 2) Risk flags ─────────────────────────────────────────────────────────────
risk_flags = []

# Pretrend
if flag_pretrend:
    risk_flags.append({
        "flag":    "⚠️  Pretrend violation",
        "detail":  "Treated and control markets show diverging slopes in PRE period. "
                   "DiD parallel trends assumption likely violated. DiD estimate discarded.",
        "level":   "warn",
    })
else:
    risk_flags.append({
        "flag":    "✅  Pretrend — clean",
        "detail":  "No significant slope divergence detected in PRE period. "
                   "Parallel trends assumption holds for DiD.",
        "level":   "ok",
    })

# Spillover
if flag_spillover:
    risk_flags.append({
        "flag":    "⚠️  Spillover contamination",
        "detail":  "Buffer Zone (mean_excess=+0.75) and Control Guardrail (mean_excess=+0.50) "
                   "show positive excess post-campaign. DiD, TBR, CUPAC invalidated. "
                   "CUPED, Synth, Bayes are spillover-robust.",
        "level":   "warn",
    })
else:
    risk_flags.append({
        "flag":    "✅  Spillover — clean",
        "detail":  "No significant excess detected in control markets post-campaign.",
        "level":   "ok",
    })

# SCM validity
scm_valid       = bool(final.loc[final["estimator"] == "Synth (all controls)", "valid"].iloc[0])
scm_rmspe_ratio = synth_all_fit.get("rmspe_lift_ratio", float("nan"))
scm_rmspe       = synth_all_fit.get("prefit_rmspe", float("nan"))
if not scm_valid or scm_rmspe_ratio > 0.25:
    risk_flags.append({
        "flag":    "⚠️  SCM pre-fit quality",
        "detail":  f"prefit_rmspe={scm_rmspe:,.0f}, RMSPE/lift ratio={scm_rmspe_ratio:.3f} — "
                   "pre-period fit too poor to trust SCM as primary. Used as signal only.",
        "level":   "warn",
    })
else:
    risk_flags.append({
        "flag":    "✅  SCM pre-fit — acceptable",
        "detail":  f"prefit_rmspe={scm_rmspe:,.0f}, RMSPE/lift ratio={scm_rmspe_ratio:.3f} — "
                   "within acceptable threshold (<0.25). SCM is a valid corroborating estimate.",
        "level":   "ok",
    })

# Model consensus / mismatch
valid_lifts = [
    float(final.loc[final["estimator"] == "CUPED (variance reduction)",       "lift_hat_total"].iloc[0]),
    float(final.loc[final["estimator"] == "Synth (all controls)",             "lift_hat_total"].iloc[0]),
    float(final.loc[final["estimator"] == "Bayes hier (partial pooling)",     "lift_hat_total"].iloc[0]),
]
consensus_spread_pct = (max(valid_lifts) - min(valid_lifts)) / abs(np.mean(valid_lifts))

if consensus_spread_pct > 0.30:
    risk_flags.append({
        "flag":    "⚠️  Model mismatch",
        "detail":  f"Valid estimators spread = {consensus_spread_pct:.1%} of mean lift — "
                   "high disagreement across methods. Investigate before scaling.",
        "level":   "warn",
    })
else:
    risk_flags.append({
        "flag":    "✅  Model consensus — tight",
        "detail":  f"CUPED / Synth / Bayes cluster within {consensus_spread_pct:.1%} of mean lift "
                   f"({min(valid_lifts):,.0f} – {max(valid_lifts):,.0f}). "
                   "Estimates are mutually corroborating.",
        "level":   "ok",
    })

# CI availability risk
missing_ci = final.loc[final["valid"] & final["ci_low"].isna(), "estimator"].tolist()
if missing_ci:
    risk_flags.append({
        "flag":    f"ℹ️  No CI for: {', '.join(missing_ci)}",
        "detail":  "These valid estimators lack bootstrap CIs. iROAS bounds computed "
                   "from CUPED only. Add posterior intervals to Bayes to strengthen bounds.",
        "level":   "info",
    })

# ── 3) Render HTML decision table ─────────────────────────────────────────────
flag_colors = {"ok": "#E8F5E9", "warn": "#FFF8E1", "info": "#E3F2FD"}
flag_text   = {"ok": "#1B5E20", "warn": "#E65100", "info": "#0D47A1"}

why_html = "".join(f"<li style='margin:4px 0'>{w}</li>" for w in why)
flag_rows = "".join(f"""
    <tr style='background:{flag_colors[f['level']]}'>
        <td style='padding:8px 12px;font-weight:500;color:{flag_text[f['level']]};
                   white-space:nowrap'>{f['flag']}</td>
        <td style='padding:8px 12px;color:#333;font-size:13px'>{f['detail']}</td>
    </tr>""" for f in risk_flags)

html = f"""
<div style='font-family:sans-serif;max-width:860px'>

  <!-- Recommendation banner -->
  <div style='background:{rec_bg};border-left:5px solid {rec_color};
              padding:16px 20px;border-radius:6px;margin-bottom:20px'>
    <div style='font-size:22px;font-weight:700;color:{rec_color};
                margin-bottom:8px'>Recommendation: {recommendation}</div>
    <ul style='margin:0;padding-left:20px;color:#333;line-height:1.7'>
      {why_html}
    </ul>
  </div>

  <!-- Key metrics strip -->
  <div style='display:flex;gap:12px;margin-bottom:20px;flex-wrap:wrap'>
    {''.join(f"""
    <div style='flex:1;min-width:130px;background:#F5F5F5;border-radius:6px;
                padding:12px 16px;text-align:center'>
      <div style='font-size:11px;color:#888;text-transform:uppercase;
                  letter-spacing:.5px;margin-bottom:4px'>{label}</div>
      <div style='font-size:18px;font-weight:600;color:#1A1A1A'>{value}</div>
      <div style='font-size:11px;color:#999;margin-top:2px'>{sub}</div>
    </div>""" for label, value, sub in [
        ("Lift (primary)",     f"{primary_lift:,.0f}",              "CUPED"),
        ("95% CI",             f"[{primary_ci_low:,.0f} — {primary_ci_high:,.0f}]", "units"),
        ("Ad spend",           f"${spend_total:,.0f}",              "treated post"),
        ("iROAS",              f"{iroas_point:.2f}x",               f"CI [{iroas_low:.2f}x — {metrics_high['iROAS']:.2f}x]"),
        ("P(iROAS>1)",         f"{breakeven_p:.1%}",                "break-even prob"),
        ("Consensus range",    f"{min(valid_lifts):,.0f}–{max(valid_lifts):,.0f}", "CUPED/Synth/Bayes"),
    ])}
  </div>

  <!-- Risk flags table -->
  <div style='font-size:13px;font-weight:600;color:#555;
              text-transform:uppercase;letter-spacing:.5px;margin-bottom:8px'>
    Risk flags
  </div>
  <table style='width:100%;border-collapse:collapse;font-size:13px;
                border:1px solid #E0E0E0;border-radius:6px;overflow:hidden'>
    <thead>
      <tr style='background:#F5F5F5'>
        <th style='padding:8px 12px;text-align:left;color:#555;
                   font-weight:600;width:240px'>Flag</th>
        <th style='padding:8px 12px;text-align:left;color:#555;
                   font-weight:600'>Detail</th>
      </tr>
    </thead>
    <tbody>{flag_rows}</tbody>
  </table>

  <!-- Footer note -->
  <div style='margin-top:14px;font-size:11px;color:#999;font-style:italic'>
    Primary estimator: CUPED (variance reduction) — only valid CI-bearing method.
    Corroborated by Synth ({synth_lift:,.0f}) and Bayes ({bayes_lift:,.0f}).
    Ground truth (simulation only): {gt_total:,.0f}.
  </div>

</div>
"""

display(HTML(html))

Flag,Detail
✅ Pretrend — clean,No significant slope divergence detected in PRE period. Parallel trends assumption holds for DiD.
⚠️ Spillover contamination,"Buffer Zone (mean_excess=+0.75) and Control Guardrail (mean_excess=+0.50) show positive excess post-campaign. DiD, TBR, CUPAC invalidated. CUPED, Synth, Bayes are spillover-robust."
✅ SCM pre-fit — acceptable,"prefit_rmspe=1,998, RMSPE/lift ratio=0.023 — within acceptable threshold (<0.25). SCM is a valid corroborating estimate."
✅ Model consensus — tight,"CUPED / Synth / Bayes cluster within 12.6% of mean lift (82,519 – 93,575). Estimates are mutually corroborating."
"ℹ️ No CI for: Synth (all controls), Bayes hier (partial pooling)",These valid estimators lack bootstrap CIs. iROAS bounds computed from CUPED only. Add posterior intervals to Bayes to strengthen bounds.


# 7.4 Conclusion: Decision-Grade Insights & Strategic Recommendation
## The Business Answer
Based on the results of this geo-experiment, the recommendation is to **STOP / PAUSE the current paid social campaign.** While the campaign successfully generated incremental sales, it did so at an inefficient cost that failed to achieve a positive return on investment.

- **Incremental Impact:** The campaign drove an estimated 82,519 incremental units (CUPED). While directionally positive across all valid models (Synth: 87.7K, Bayes: 93.6K), this lift was insufficient to offset the $945,700 in ad spend.

- **Efficiency (iROAS):** The primary incremental Return on Ad Spend (iROAS) is 0.09x, meaning every 1 USD spent returned only 0.09 USD in revenue. Even at the most optimistic 95% confidence bound (0.14x), the campaign remains significantly below the 1.0x break-even threshold.

- **Confidence:** We estimate a 0.0% probability that this campaign reached break-even. The decision to stop is supported by tight model consensus across three distinct causal methodologies.

## Technical Rigor & Trust Gates
This measurement pipeline utilized a multi-model consensus framework to ensure the recommendation was not a byproduct of a single estimator's bias:

1. **Causal Validity:** We successfully identified and adjusted for control group contamination (spillover) that would have otherwise biased standard DiD or TBR estimates.

2. **Precision Engineering:** By deploying CUPED (Variance Reduction), we narrowed the confidence intervals by 99.4%, transforming a "noisy" result into a decision-grade metric.

3. **Triangulation:** The proximity of the Bayesian Hierarchical estimate (+2.2% from ground truth) to our primary estimator provides high "real-world" confidence in the validity of our synthetic data generation and recovery logic.

## Strategic Next Steps
**Creative/Targeting Pivot:** Given the low iROAS but confirmed incremental lift, the focus should shift from **"scaling spend" to "optimizing unit economics."**

- **Pipeline Evolution:** Future iterations will incorporate Posterior Credible Intervals for the Bayesian model to provide even more robust uncertainty bounds for automated spend-bidding systems.